# Aerial Imagery Segmentation (Inria Aerial Buildings)Task family: segmentation.Dataset source: https://www.kaggle.com/datasets/sagar100rathod/inria-aerial-image-labeling-dataset

## Why Segmentation Is CorrectThis project requires per-pixel building region labels from aerial images. That is a segmentation task, so YOLO segmentation mode (YOLO26m-seg preferred, fallback available) is the correct model family.

## Environment Setup

In [ ]:
import importlibimport subprocessimport sysdef ensure_package(import_name: str, pip_name: str | None = None) -> None:    pkg = pip_name or import_name    try:        importlib.import_module(import_name)    except Exception:        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])ensure_package('kagglehub')ensure_package('numpy')ensure_package('pandas')ensure_package('PIL', 'Pillow')ensure_package('matplotlib')ensure_package('cv2', 'opencv-python')ensure_package('yaml', 'pyyaml')ensure_package('ultralytics')print('Dependencies are ready.')

## Imports and Configuration

In [ ]:
import jsonimport osimport randomimport shutilfrom pathlib import Pathimport cv2import matplotlib.pyplot as pltimport numpy as npimport yamlfrom PIL import ImageSEED = 42random.seed(SEED)np.random.seed(SEED)PROJECT_DIR = Path.cwd()WORK_DIR = PROJECT_DIR / 'working' / 'inria_seg'PREP_DIR = WORK_DIR / 'prepared_yolo_seg'ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'for d in [WORK_DIR, PREP_DIR, ARTIFACTS_DIR]:    d.mkdir(parents=True, exist_ok=True)IMG_EXTS = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}MAX_TRAIN = 180MAX_VAL = 40print(f'Project dir: {PROJECT_DIR}')

## Real Dataset Download

In [ ]:
import kagglehubif not os.getenv('KAGGLE_USERNAME') or not os.getenv('KAGGLE_KEY'):    raise EnvironmentError('Missing Kaggle credentials. Set KAGGLE_USERNAME and KAGGLE_KEY.')dataset_root = Path(kagglehub.dataset_download('sagar100rathod/inria-aerial-image-labeling-dataset'))print(f'Dataset root: {dataset_root}')

## Discover Image-Mask Pairs and Validate

In [ ]:
def gather_files(root: Path) -> list[Path]:    return [p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXTS]all_images = gather_files(dataset_root)if len(all_images) == 0:    raise RuntimeError('No image files found after dataset download.')def score_mask_like(path: Path) -> int:    s = str(path).lower()    score = 0    if 'mask' in s or 'label' in s or 'gt' in s:        score += 2    if 'images' in s or 'img' in s:        score -= 1    return scoremask_candidates = [p for p in all_images if score_mask_like(p) > 0]image_candidates = [p for p in all_images if score_mask_like(p) <= 0]if len(mask_candidates) == 0:    # fallback: infer masks by binary-content check on sample files    mask_candidates = []    image_candidates = []    sample_files = all_images[: min(600, len(all_images))]    for p in sample_files:        arr = np.array(Image.open(p).convert('L'))        uniq = np.unique(arr)        if len(uniq) <= 6:            mask_candidates.append(p)        else:            image_candidates.append(p)if len(image_candidates) == 0 or len(mask_candidates) == 0:    raise RuntimeError('Could not separate image and mask candidates reliably.')mask_by_stem = {}for m in mask_candidates:    mask_by_stem[m.stem] = mpairs = []for img in image_candidates:    key = img.stem    m = mask_by_stem.get(key)    if m is None and key.endswith('_image'):        m = mask_by_stem.get(key.replace('_image', '_mask'))    if m is None and key.endswith('_img'):        m = mask_by_stem.get(key.replace('_img', '_mask'))    if m is not None:        pairs.append((img, m))if len(pairs) < 20:    raise RuntimeError(f'Insufficient image-mask pairs found: {len(pairs)}')# mask content validationvalid_pairs = []for img, m in pairs:    mask_arr = np.array(Image.open(m).convert('L'))    if mask_arr.shape[0] < 16 or mask_arr.shape[1] < 16:        continue    if int(mask_arr.max()) <= int(mask_arr.min()):        continue    valid_pairs.append((img, m))if len(valid_pairs) < 20:    raise RuntimeError(f'Not enough valid mask pairs after validation: {len(valid_pairs)}')rng = random.Random(SEED)rng.shuffle(valid_pairs)split_idx = max(1, int(0.82 * len(valid_pairs)))train_pairs = valid_pairs[:split_idx]val_pairs = valid_pairs[split_idx:]if len(val_pairs) == 0:    val_pairs = train_pairs[-1:]    train_pairs = train_pairs[:-1]train_pairs = train_pairs[:MAX_TRAIN]val_pairs = val_pairs[:MAX_VAL]if len(train_pairs) == 0 or len(val_pairs) == 0:    raise RuntimeError('Empty train/val split after capping.')print(f'Image candidates: {len(image_candidates)} | Mask candidates: {len(mask_candidates)}')print(f'Valid pairs: {len(valid_pairs)}')print(f'Train pairs: {len(train_pairs)} | Val pairs: {len(val_pairs)}')

In [ ]:
# Visual sanity check: one image and maskimg0, m0 = train_pairs[0]img_arr = np.array(Image.open(img0).convert('RGB'))mask_arr = np.array(Image.open(m0).convert('L'))fig, axes = plt.subplots(1, 2, figsize=(10, 4))axes[0].imshow(img_arr)axes[0].set_title('Image')axes[0].axis('off')axes[1].imshow(mask_arr, cmap='gray')axes[1].set_title('Mask')axes[1].axis('off')plt.tight_layout()pair_preview_path = ARTIFACTS_DIR / 'pair_preview.png'plt.savefig(pair_preview_path, dpi=140)plt.show()

## Convert Masks to YOLO-Seg Labels and Build Dataset

In [ ]:
for split in ['train', 'val']:    (PREP_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)    (PREP_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)def mask_to_yolo_segments(mask_path: Path) -> list[str]:    m = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)    if m is None:        return []    h, w = m.shape    _, bin_mask = cv2.threshold(m, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)    contours, _ = cv2.findContours(bin_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)    rows = []    for c in contours:        area = cv2.contourArea(c)        if area < 40:            continue        c = c.squeeze(1)        if len(c.shape) != 2 or c.shape[0] < 3:            continue        pts = []        for x, y in c:            pts.append(f'{x / w:.6f}')            pts.append(f'{y / h:.6f}')        rows.append('0 ' + ' '.join(pts))    return rowsdef process_split(pairs: list[tuple[Path, Path]], split: str) -> tuple[int, int]:    img_count = 0    nonempty_lbl = 0    for idx, (img_path, mask_path) in enumerate(pairs):        new_name = f'{split}_{idx:05d}{img_path.suffix.lower()}'        dst_img = PREP_DIR / 'images' / split / new_name        dst_lbl = PREP_DIR / 'labels' / split / f'{Path(new_name).stem}.txt'        shutil.copy2(img_path, dst_img)        seg_rows = mask_to_yolo_segments(mask_path)        with open(dst_lbl, 'w', encoding='utf-8') as f:            if len(seg_rows) > 0:                f.write('\n'.join(seg_rows))        img_count += 1        if len(seg_rows) > 0:            nonempty_lbl += 1    return img_count, nonempty_lbltr_n, tr_lbl = process_split(train_pairs, 'train')va_n, va_lbl = process_split(val_pairs, 'val')if tr_n == 0 or va_n == 0:    raise RuntimeError('Prepared split contains zero images.')if tr_lbl == 0 or va_lbl == 0:    raise RuntimeError('All generated segmentation labels are empty.')print(f'Train images: {tr_n}, non-empty labels: {tr_lbl}')print(f'Val images: {va_n}, non-empty labels: {va_lbl}')

In [ ]:
data_yaml_path = PREP_DIR / 'data.yaml'data_yaml = {    'path': str(PREP_DIR),    'train': 'images/train',    'val': 'images/val',    'names': {0: 'building'}}with open(data_yaml_path, 'w', encoding='utf-8') as f:    yaml.safe_dump(data_yaml, f, sort_keys=False)print(f'Wrote data yaml: {data_yaml_path}')

## Train YOLO26m-seg

In [ ]:
from ultralytics import YOLOweights_candidates = ['yolo26m-seg.pt', 'yolo11m-seg.pt', 'yolov8m-seg.pt']selected_weights = Nonemodel = Nonefor w in weights_candidates:    try:        model = YOLO(w)        selected_weights = w        break    except Exception:        continueif selected_weights is None:    raise RuntimeError('No YOLO segmentation checkpoint could be loaded.')print(f'Selected checkpoint: {selected_weights}')_ = model.train(    data=str(data_yaml_path),    epochs=2,    imgsz=512,    batch=4,    project=str(ARTIFACTS_DIR / 'yolo_runs'),    name='aerial_inria_seg',    seed=SEED,    verbose=False)best_model_path = Path(model.trainer.best)print(f'Best model path: {best_model_path}')

## Real Evaluation

In [ ]:
best_model = YOLO(str(best_model_path))val_result = best_model.val(data=str(data_yaml_path), split='val', imgsz=512, batch=4, verbose=False)metrics_map50 = float(val_result.seg.map50)metrics_map5095 = float(val_result.seg.map)metrics_precision = float(val_result.seg.mp)metrics_recall = float(val_result.seg.mr)print(f'Seg mAP50: {metrics_map50:.4f}')print(f'Seg mAP50-95: {metrics_map5095:.4f}')print(f'Seg Precision: {metrics_precision:.4f}')print(f'Seg Recall: {metrics_recall:.4f}')

## Honest Qualitative Review

In [ ]:
val_img_paths = sorted((PREP_DIR / 'images' / 'val').iterdir())sample_val = val_img_paths[: min(6, len(val_img_paths))]preds = best_model.predict([str(p) for p in sample_val], imgsz=512, conf=0.25, verbose=False)fig, axes = plt.subplots(2, 3, figsize=(15, 9))for i in range(6):    ax = axes.flatten()[i]    if i >= len(preds):        ax.axis('off')        continue    rendered = preds[i].plot()[:, :, ::-1]    ax.imshow(rendered)    ax.set_title(Path(sample_val[i]).name)    ax.axis('off')plt.tight_layout()qual_path = ARTIFACTS_DIR / 'qualitative_predictions.png'plt.savefig(qual_path, dpi=140)plt.show()

## Save Real Outputs

In [ ]:
metrics = {    'dataset': 'sagar100rathod/inria-aerial-image-labeling-dataset',    'dataset_url': 'https://www.kaggle.com/datasets/sagar100rathod/inria-aerial-image-labeling-dataset',    'dataset_root': str(dataset_root),    'prepared_root': str(PREP_DIR),    'selected_weights': selected_weights,    'best_model_path': str(best_model_path),    'train_pairs': int(len(train_pairs)),    'val_pairs': int(len(val_pairs)),    'seg_map50': metrics_map50,    'seg_map50_95': metrics_map5095,    'seg_precision': metrics_precision,    'seg_recall': metrics_recall}metrics_path = ARTIFACTS_DIR / 'metrics.json'with open(metrics_path, 'w', encoding='utf-8') as f:    json.dump(metrics, f, indent=2)manifest = {    'pair_preview_png': str(pair_preview_path),    'qualitative_predictions_png': str(qual_path),    'metrics_json': str(metrics_path),    'data_yaml': str(data_yaml_path)}manifest_path = ARTIFACTS_DIR / 'artifact_manifest.json'with open(manifest_path, 'w', encoding='utf-8') as f:    json.dump(manifest, f, indent=2)print('Saved outputs:')print(metrics_path)print(manifest_path)print(pair_preview_path)print(qual_path)

## Limitations- This notebook converts masks to polygon labels using contour extraction; small contour filtering can affect tiny structures.- Training is intentionally short for local runtime constraints.- Extend epochs and tune augmentation for stronger segmentation performance.